In [5]:
import pandas as pd
import numpy as np
import os

In [6]:
#UPLOADING FILES FOR BIRD_SPECIES_OBSERAVTION_FOREST

folder_path = r'C:\Users\addyu\OneDrive\Desktop\LABMENTIX\BIRD SPECIES OBSERVATION\FOREST'

# Loading all .csv files in that folder
csv_files = [file for file in os.listdir(folder_path) if file.endswith('.csv')]

# Loading into a dictionary of DataFrames
dfs = {}
for file in csv_files:
    name = file.replace('.csv', '')
    file_path = os.path.join(folder_path, file)
    df = pd.read_csv(file_path)
    dfs[name] = df
    print(f" Loaded: {file} | Shape: {df.shape}")


 Loaded: ANTI_FOREST.csv | Shape: (999, 29)
 Loaded: CATO_FOREST.csv | Shape: (999, 29)
 Loaded: CHOH_FOREST.csv | Shape: (2202, 29)
 Loaded: GWMP_FOREST.csv | Shape: (999, 29)
 Loaded: HAFE_FOREST.csv | Shape: (999, 29)
 Loaded: MANA_FOREST.csv | Shape: (999, 29)
 Loaded: MONO_FOREST.csv | Shape: (999, 29)
 Loaded: NACE_FOREST.csv | Shape: (999, 29)
 Loaded: PWRI_FOREST.csv | Shape: (2463, 29)
 Loaded: ROCR_FOREST.csv | Shape: (999, 29)
 Loaded: WOTR_FOREST.csv | Shape: (999, 29)


In [7]:
#UPLOADING FILES FOR BIRD_SPECIES_OBSERAVTION_GRASSLAND

grassland_path = r'C:\Users\addyu\OneDrive\Desktop\LABMENTIX\BIRD SPECIES OBSERVATION\GRASSLAND'

# Loading all .csv files in GRASSLAND folder
grassland_csvs = [file for file in os.listdir(grassland_path) if file.endswith('.csv')]

# Dictionary to store DataFrames
grassland_dfs = {}

for file in grassland_csvs:
    name = file.replace('.csv', '')
    file_path = os.path.join(grassland_path, file)
    df = pd.read_csv(file_path)
    grassland_dfs[name] = df
    print(f" Loaded: {file} | Shape: {df.shape}")
    

 Loaded: ANTI_GRASSLAND.csv | Shape: (3588, 29)
 Loaded: CATO.csv | Shape: (999, 29)
 Loaded: CHOH.csv | Shape: (999, 29)
 Loaded: GWMP.csv | Shape: (999, 29)
 Loaded: HAFE_GRASSLAND.csv | Shape: (999, 29)
 Loaded: MANA_GRASSLAND.csv | Shape: (1811, 29)
 Loaded: MONO_GRASSLAND.csv | Shape: (3015, 29)
 Loaded: NACE.csv | Shape: (999, 29)
 Loaded: PWRI.csv | Shape: (999, 29)
 Loaded: ROCR.csv | Shape: (999, 29)
 Loaded: WOTR.csv | Shape: (999, 29)


In [8]:
forest_path = r'C:\Users\addyu\OneDrive\Desktop\LABMENTIX\BIRD SPECIES OBSERVATION\FOREST'
grassland_path = r'C:\Users\addyu\OneDrive\Desktop\LABMENTIX\BIRD SPECIES OBSERVATION\GRASSLAND'

def load_csvs(folder_path):
    files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]
    dataframes = {}
    for f in files:
        df = pd.read_csv(os.path.join(folder_path, f))
        name = f.replace('.csv', '')
        dataframes[name] = df
    return dataframes

# Loading datasets
forest_dfs = load_csvs(forest_path)
grassland_dfs = load_csvs(grassland_path)

#Combining for cleaning
all_datasets = {**forest_dfs, **grassland_dfs}

#Master Cleaning Loop
cleaned_datasets = {}

for name, df in all_datasets.items():
    print(f"\n Cleaning: {name}")
    
    # Dropping if dataset is fully empty (7 empty grassland sheets)
    if df.dropna(how='all').shape[0] == 0:
        print(f" Skipped: {name} is completely empty.")
        continue

    # Stripping and standardizing column names
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

    # Dropping fully null columns
    df.dropna(axis=1, how='all', inplace=True)

    # Dropping duplicates
    df.drop_duplicates(inplace=True)




 Cleaning: ANTI_FOREST

 Cleaning: CATO_FOREST

 Cleaning: CHOH_FOREST

 Cleaning: GWMP_FOREST

 Cleaning: HAFE_FOREST

 Cleaning: MANA_FOREST

 Cleaning: MONO_FOREST

 Cleaning: NACE_FOREST

 Cleaning: PWRI_FOREST

 Cleaning: ROCR_FOREST

 Cleaning: WOTR_FOREST

 Cleaning: ANTI_GRASSLAND

 Cleaning: CATO
 Skipped: CATO is completely empty.

 Cleaning: CHOH
 Skipped: CHOH is completely empty.

 Cleaning: GWMP
 Skipped: GWMP is completely empty.

 Cleaning: HAFE_GRASSLAND

 Cleaning: MANA_GRASSLAND

 Cleaning: MONO_GRASSLAND

 Cleaning: NACE
 Skipped: NACE is completely empty.

 Cleaning: PWRI
 Skipped: PWRI is completely empty.

 Cleaning: ROCR
 Skipped: ROCR is completely empty.

 Cleaning: WOTR
 Skipped: WOTR is completely empty.


In [52]:
#BIRD_SPECIES_OBSERAVTION_FOREST

def enrich_with_derived_columns(df):
    # Standardize column names
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

    # --- DATE FEATURES ---
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df['month'] = df['date'].dt.month
    df['day_of_week'] = df['date'].dt.day_name()

    # Season from month
    def get_season(month):
        if pd.isna(month): return None
        if month in [3, 4, 5]: return 'Spring'
        elif month in [6, 7, 8]: return 'Summer'
        elif month in [9, 10, 11]: return 'Autumn'
        else: return 'Winter'
    df['season'] = df['month'].apply(get_season)

    # --- TIME FEATURES ---
    df['start_time'] = pd.to_datetime(df['start_time'], errors='coerce')
    df['end_time'] = pd.to_datetime(df['end_time'], errors='coerce')
    df['observation_duration'] = (df['end_time'] - df['start_time']).dt.total_seconds() / 60
    df['observation_hour_start'] = df['start_time'].dt.hour
    df['observation_hour_end'] = df['end_time'].dt.hour

    # --- SEX CATEGORY ---
    df['sex_category'] = df['sex'].fillna('Undetermined')

    # --- PLOT & OBSERVER BASED ---
    df['plot_activity_score'] = df.groupby('plot_name')['plot_name'].transform('count')
    df['observer_freq'] = df.groupby('observer')['observer'].transform('count')

    # --- SPECIES RICHNESS FLAG ---
    df['species_richness_flag'] = (df['initial_three_min_cnt'] > 5).astype(int)

    # --- WEATHER COMBINATION ---
    df['weather_combination'] = (
        df['sky'].fillna('') + ' | ' +
        df['wind'].fillna('') + ' | Temp: ' +
        df['temperature'].fillna('NA').astype(str)
    )

    # --- DISTANCE CATEGORY ---
    def classify_distance(dist):
        if pd.isna(dist):
            return 'Unknown'
        dist_str = str(dist).lower()
        if '<=' in dist_str or '≤' in dist_str or '50' in dist_str:
            return '<= 50 meters'
        else:
            return '> 50 meters'
    df['distance_category'] = df['distance'].apply(classify_distance)

    return df


In [53]:
# Apply enrichment function to all forest datasets
enhanced_forest_dfs = {}
for name, df in forest_dfs.items():
    print(f"Enhancing {name}...")
    enhanced_forest_dfs[name] = enrich_with_derived_columns(df)


Enhancing ANTI_FOREST...
Enhancing CATO_FOREST...
Enhancing CHOH_FOREST...
Enhancing GWMP_FOREST...
Enhancing HAFE_FOREST...
Enhancing MANA_FOREST...
Enhancing MONO_FOREST...
Enhancing NACE_FOREST...
Enhancing PWRI_FOREST...
Enhancing ROCR_FOREST...
Enhancing WOTR_FOREST...


In [54]:
for name, df in enhanced_forest_dfs.items():
    print(f"\n Columns in {name}:")
    print(list(df.columns))



 Columns in ANTI_FOREST:
['admin_unit_code', 'site_name', 'plot_name', 'location_type', 'year', 'date', 'start_time', 'end_time', 'observer', 'visit', 'interval_length', 'id_method', 'distance', 'flyover_observed', 'sex', 'common_name', 'scientific_name', 'acceptedtsn', 'npstaxoncode', 'aou_code', 'pif_watchlist_status', 'regional_stewardship_status', 'temperature', 'humidity', 'sky', 'wind', 'disturbance', 'initial_three_min_cnt', 'month', 'day_of_week', 'season', 'observation_duration', 'observation_hour_start', 'observation_hour_end', 'sex_category', 'plot_activity_score', 'observer_freq', 'species_richness_flag', 'weather_combination', 'distance_category', 'observer_frequency']

 Columns in CATO_FOREST:
['admin_unit_code', 'site_name', 'plot_name', 'location_type', 'year', 'date', 'start_time', 'end_time', 'observer', 'visit', 'interval_length', 'id_method', 'distance', 'flyover_observed', 'sex', 'common_name', 'scientific_name', 'acceptedtsn', 'npstaxoncode', 'aou_code', 'pif_wat

In [40]:
#BIRD_SPECIES_OBSERVATION_GRASSLAND

# 1. List of datasets to KEEP
keep_grassland = ['ANTI_GRASSLAND', 'HAFE_GRASSLAND', 'MANA_GRASSLAND', 'MONO_GRASSLAND']

# 2. Dropping the rest from grassland_dfs
grassland_dfs = {k: v for k, v in grassland_dfs.items() if k in keep_grassland}

In [41]:
def enrich_with_derived_columns(df):
    # Standardize column names
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

    # --- DATE FEATURES ---
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df['month'] = df['date'].dt.month
    df['day_of_week'] = df['date'].dt.day_name()

    def get_season(month):
        if pd.isna(month): return None
        if month in [3, 4, 5]: return 'Spring'
        elif month in [6, 7, 8]: return 'Summer'
        elif month in [9, 10, 11]: return 'Autumn'
        else: return 'Winter'
    df['season'] = df['month'].apply(get_season)

    # --- TIME FEATURES ---
    df['start_time'] = pd.to_datetime(df['start_time'], errors='coerce')
    df['end_time'] = pd.to_datetime(df['end_time'], errors='coerce')
    df['observation_duration'] = (df['end_time'] - df['start_time']).dt.total_seconds() / 60
    df['observation_hour_start'] = df['start_time'].dt.hour
    df['observation_hour_end'] = df['end_time'].dt.hour

    def time_period(hour):
        if pd.isna(hour): return 'Unknown'
        if 4 <= hour < 10: return 'Morning'
        elif 10 <= hour < 15: return 'Midday'
        elif 15 <= hour < 19: return 'Evening'
        else: return 'Night'
    df['observation_time_period'] = df['observation_hour_start'].apply(time_period)

    # --- SEX CATEGORY ---
    df['sex_category'] = df['sex'].fillna('Undetermined')

    # --- ACTIVITY + OBSERVER ---
    df['plot_activity_score'] = df.groupby('plot_name')['plot_name'].transform('count')
    df['observer_freq'] = df.groupby('observer')['observer'].transform('count')

    # --- SPECIES RICHNESS FLAG ---
    df['species_richness_flag'] = (df['initial_three_min_cnt'] > 5).astype(int)

    # --- WEATHER COMBINATION ---
    df['weather_combination'] = (
        df['sky'].fillna('') + ' | ' +
        df['wind'].fillna('') + ' | Temp: ' +
        df['temperature'].fillna('NA').astype(str)
    )

    # --- DISTANCE CATEGORY ---
    def classify_distance(dist):
        if pd.isna(dist): return 'Unknown'
        dist_str = str(dist).lower()
        if '<=' in dist_str or '≤' in dist_str or '50' in dist_str:
            return '<= 50 meters'
        else:
            return '> 50 meters'
    df['distance_category'] = df['distance'].apply(classify_distance)

    return df


In [42]:
# Ensure only relevant datasets are used
target_grassland_dfs = {
    name: df for name, df in grassland_dfs.items()
    if name in ['ANTI_GRASSLAND', 'HAFE_GRASSLAND', 'MANA_GRASSLAND', 'MONO_GRASSLAND']
}

# Apply and store enriched results
enhanced_grassland_dfs = {}
for name, df in target_grassland_dfs.items():
    print(f" Enriching: {name}")
    enhanced_grassland_dfs[name] = enrich_with_derived_columns(df)


 Enriching: ANTI_GRASSLAND
 Enriching: HAFE_GRASSLAND
 Enriching: MANA_GRASSLAND
 Enriching: MONO_GRASSLAND


In [43]:
for name, df in enhanced_grassland_dfs.items():
    print(f"\n Columns in {name}:")
    print(list(df.columns))



 Columns in ANTI_GRASSLAND:
['admin_unit_code', 'plot_name', 'location_type', 'year', 'date', 'start_time', 'end_time', 'observer', 'visit', 'interval_length', 'id_method', 'distance', 'flyover_observed', 'sex', 'common_name', 'scientific_name', 'acceptedtsn', 'taxoncode', 'aou_code', 'pif_watchlist_status', 'regional_stewardship_status', 'temperature', 'humidity', 'sky', 'wind', 'disturbance', 'previously_obs', 'initial_three_min_cnt', 'month', 'day_of_week', 'season', 'observation_duration', 'observation_hour_start', 'observation_hour_end', 'observation_time_period', 'sex_category', 'plot_activity_score', 'observer_freq', 'species_richness_flag', 'weather_combination', 'distance_category']

 Columns in HAFE_GRASSLAND:
['admin_unit_code', 'plot_name', 'location_type', 'year', 'date', 'start_time', 'end_time', 'observer', 'visit', 'interval_length', 'id_method', 'distance', 'flyover_observed', 'sex', 'common_name', 'scientific_name', 'acceptedtsn', 'taxoncode', 'aou_code', 'pif_watchl

In [44]:
# CHECKING NULLS AND DUPLICATES FOR DICTIONARY OF DATAFRAMES

def check_nulls_duplicates(dataset_dict, label):
    print(f"\n Checking {label} datasets...\n" + "-"*50)
    for name, df in dataset_dict.items():
        print(f"\n Dataset: {name}")
        print(f" Shape: {df.shape}")
        
        # Null value count and %
        null_counts = df.isnull().sum()
        
        print("\n Null Value Counts:")
        print(null_counts[null_counts > 0])
        
        
        # Duplicate rows
        dup_count = df.duplicated().sum()
        print(f"\n Duplicate Rows: {dup_count}")
        print("-" * 50)

# Run checks
check_nulls_duplicates(enhanced_forest_dfs, "FOREST")
check_nulls_duplicates(enhanced_grassland_dfs, "GRASSLAND")



 Checking FOREST datasets...
--------------------------------------------------

 Dataset: ANTI_FOREST
 Shape: (334, 41)

 Null Value Counts:
admin_unit_code                  1
site_name                        1
plot_name                        1
location_type                    1
year                             1
date                             1
start_time                     334
end_time                       334
observer                         1
visit                            1
interval_length                  1
id_method                        1
distance                         6
flyover_observed                 1
sex                            222
common_name                      1
scientific_name                  1
acceptedtsn                      2
npstaxoncode                     1
aou_code                         1
pif_watchlist_status             1
regional_stewardship_status      1
temperature                      1
humidity                         1
sky              

In [45]:
# REMOVING DUPLICATES FROM ALL ENHANCED FOREST DATASETS

for name in enhanced_forest_dfs:
    before = enhanced_forest_dfs[name].shape[0]
    enhanced_forest_dfs[name] = enhanced_forest_dfs[name].drop_duplicates()
    after = enhanced_forest_dfs[name].shape[0]
    print(f" {name}: Removed {before - after} duplicate rows")

# REMOVING DUPLICATES FROM ALL ENHANCED GRASSLAND DATASETS

for name in enhanced_grassland_dfs:
    before = enhanced_grassland_dfs[name].shape[0]
    enhanced_grassland_dfs[name] = enhanced_grassland_dfs[name].drop_duplicates()
    after = enhanced_grassland_dfs[name].shape[0]
    print(f" {name}: Removed {before - after} duplicate rows")


 ANTI_FOREST: Removed 0 duplicate rows
 CATO_FOREST: Removed 0 duplicate rows
 CHOH_FOREST: Removed 0 duplicate rows
 GWMP_FOREST: Removed 0 duplicate rows
 HAFE_FOREST: Removed 0 duplicate rows
 MANA_FOREST: Removed 0 duplicate rows
 MONO_FOREST: Removed 0 duplicate rows
 NACE_FOREST: Removed 0 duplicate rows
 PWRI_FOREST: Removed 0 duplicate rows
 ROCR_FOREST: Removed 0 duplicate rows
 WOTR_FOREST: Removed 0 duplicate rows
 ANTI_GRASSLAND: Removed 0 duplicate rows
 HAFE_GRASSLAND: Removed 0 duplicate rows
 MANA_GRASSLAND: Removed 0 duplicate rows
 MONO_GRASSLAND: Removed 0 duplicate rows


In [46]:
def print_dataset_shapes(dataset_dict, label):
    print(f"\n Final Rows & Columns for {label.upper()} Datasets\n" + "-"*50)
    for name, df in dataset_dict.items():
        print(f"{name}: {df.shape[0]} rows × {df.shape[1]} columns")

# Show shapes for forest and grassland datasets
print_dataset_shapes(enhanced_forest_dfs, "forest")
print_dataset_shapes(enhanced_grassland_dfs, "grassland")



 Final Rows & Columns for FOREST Datasets
--------------------------------------------------
ANTI_FOREST: 334 rows × 41 columns
CATO_FOREST: 806 rows × 41 columns
CHOH_FOREST: 2202 rows × 41 columns
GWMP_FOREST: 387 rows × 42 columns
HAFE_FOREST: 423 rows × 41 columns
MANA_FOREST: 466 rows × 41 columns
MONO_FOREST: 371 rows × 41 columns
NACE_FOREST: 685 rows × 42 columns
PWRI_FOREST: 2463 rows × 41 columns
ROCR_FOREST: 290 rows × 41 columns
WOTR_FOREST: 128 rows × 41 columns

 Final Rows & Columns for GRASSLAND Datasets
--------------------------------------------------
ANTI_GRASSLAND: 3130 rows × 41 columns
HAFE_GRASSLAND: 109 rows × 41 columns
MANA_GRASSLAND: 1436 rows × 41 columns
MONO_GRASSLAND: 2152 rows × 41 columns


In [48]:
#Merge all into one

forest_df = pd.concat(enhanced_forest_dfs.values(), ignore_index=True)
print("Combined Forest Data Shape:", forest_df.shape)
forest_df.head(3)

Combined Forest Data Shape: (8555, 42)


,admin_unit_code,site_name,plot_name,location_type,year,date,start_time,end_time,observer,visit,...,observation_hour_start,observation_hour_end,sex_category,plot_activity_score,observer_freq,species_richness_flag,weather_combination,distance_category,observer_frequency,sub_unit_code
0,ANTI,ANTI 1,ANTI-0036,Forest,2018.0,2018-05-22,NaT,NaT,Elizabeth Oswald,1.0,...,NaN,NaN,Undetermined,34.0,127.0,0,Cloudy/Overcast | Calm (< 1 mph) smoke rises v...,<= 50 meters,127.0,NaN
1,ANTI,ANTI 1,ANTI-0036,Forest,2018.0,2018-05-22,NaT,NaT,Elizabeth Oswald,1.0,...,NaN,NaN,Undetermined,34.0,127.0,0,Cloudy/Overcast | Calm (< 1 mph) smoke rises v...,<= 50 meters,127.0,NaN
2,ANTI,ANTI 1,ANTI-0036,Forest,2018.0,2018-05-22,NaT,NaT,Elizabeth Oswald,1.0,...,NaN,NaN,Undetermined,34.0,127.0,0,Cloudy/Overcast | Calm (< 1 mph) smoke rises v...,<= 50 meters,127.0,NaN


In [ ]:
#EDA

#BIRD_SPECIES_OBSERVATION_FOREST

In [49]:
#Q1: What are the most common months and seasons of bird observation?

# Ensure 'date' column is datetime and derive 'month' and 'season'
forest_df['date'] = pd.to_datetime(forest_df['date'], errors='coerce')
forest_df['month'] = forest_df['date'].dt.month

# Map each month to a season
forest_df['season'] = forest_df['month'].map({
    12: 'Winter', 1: 'Winter', 2: 'Winter',
    3: 'Spring', 4: 'Spring', 5: 'Spring',
    6: 'Summer', 7: 'Summer', 8: 'Summer',
    9: 'Autumn', 10: 'Autumn', 11: 'Autumn'
})

# Count of observations by month
month_counts = forest_df['month'].value_counts().sort_index()

# Count of observations by season
season_counts = forest_df['season'].value_counts().reindex(['Spring', 'Summer', 'Autumn', 'Winter'])

# Display counts (you can print or explore as needed)
print("Observations by Month:")
print(month_counts)

print("Observations by Season:")
print(season_counts)


Observations by Month:
1.0      222
2.0      341
3.0      268
4.0      468
5.0     2240
6.0     2352
7.0      533
8.0      437
9.0      483
10.0     506
11.0     189
12.0     507
Name: month, dtype: int64
Observations by Season:
Spring    2976
Summer    3322
Autumn    1178
Winter    1070
Name: season, dtype: int64


In [50]:
#Q2: What are the top 10 most commonly observed species?

# Count the number of occurrences per bird species
top_species = forest_df['common_name'].value_counts().head(10)

# Display top 10 species
print("Top 10 Most Commonly Observed Species:")
print(top_species)


Top 10 Most Commonly Observed Species:
Red-eyed Vireo             694
Carolina Wren              646
Northern Cardinal          595
Eastern Tufted Titmouse    541
Eastern Wood-Pewee         486
Acadian Flycatcher         436
Red-bellied Woodpecker     398
Blue-gray Gnatcatcher      310
Ovenbird                   303
American Robin             294
Name: common_name, dtype: int64


In [56]:
#Q4:What are the most used identification methods?

# Count how often each ID method is used
id_method_counts = forest_df['id_method'].value_counts()

# Display results
print("Most Used Identification Methods:")
print(id_method_counts)



Most Used Identification Methods:
Singing          5430
Calling          2677
Visualization     438
Name: id_method, dtype: int64


In [63]:
#Q5: What are the most frequent sky and wind conditions, and what’s the average temperature & humidity?

# Count most common sky conditions
sky_counts = forest_df['sky'].value_counts()

# Count most common wind conditions
wind_counts = forest_df['wind'].value_counts()

# Calculate average temperature and humidity
avg_temp = forest_df['temperature'].mean()
avg_humidity = forest_df['humidity'].mean()

# Display results
print("Most Frequent Sky Conditions:")
print(sky_counts)

print("Most Frequent Wind Conditions:")
print(wind_counts)

print(f"\Average Temperature: {avg_temp:.2f}°C")
print(f"Average Humidity: {avg_humidity:.2f}%")


Most Frequent Sky Conditions:
Partly Cloudy          2989
Clear or Few Clouds    2976
Cloudy/Overcast        1877
Fog                     437
Mist/Drizzle            267
Name: sky, dtype: int64
Most Frequent Wind Conditions:
Light air movement (1-3 mph) smoke drifts     4102
Calm (< 1 mph) smoke rises vertically         3090
Light breeze (4-7 mph) wind felt on face      1216
Gentle breeze (8-12 mph), leaves in motion     138
Name: wind, dtype: int64
\Average Temperature: 21.87°C
Average Humidity: 77.76%


In [60]:
#Q6: What is the distribution of observation distances (≤50m vs >50m)?

# Make sure the distance column is numeric (if not already)
forest_df['distance'] = pd.to_numeric(forest_df['distance'], errors='coerce')

# Create distance category column (already created during cleaning, but re-confirm)
forest_df['distance_category'] = forest_df['distance'].apply(
    lambda x: '≤50m' if pd.notnull(x) and x <= 50 else '>50m'
)

# Count of observations in each distance category
distance_counts = forest_df['distance_category'].value_counts()

# Display result
print("Observation Distance Distribution:")
print(distance_counts)


Observation Distance Distribution:
>50m    8555
Name: distance_category, dtype: int64


In [65]:
#Q7: How many observations are on the PIF Watchlist?

# Check unique values to confirm format
print("Unique values in 'pif_watchlist status':")
print(forest_df['pif_watchlist_status'].dropna().unique())

# Count how many observations are on the PIF Watchlist
pif_watchlist_counts = forest_df['pif_watchlist_status'].value_counts()

# Display results
print("Observations on the PIF Watchlist status:")
print(pif_watchlist_counts)


Unique values in 'pif_watchlist status':
[False True]
Observations on the PIF Watchlist status:
False    8208
True      338
Name: pif_watchlist_status, dtype: int64


In [68]:
#Q8: How many birds are flagged as high regional stewardship priority? 

# Check unique values in stewardship column
print("Unique values in 'regional stewardship status':")
print(forest_df['regional_stewardship_status'].dropna().unique())

# Count number of observations flagged as high stewardship
stewardship_counts = forest_df['regional_stewardship_status'].value_counts()

# Display results
print("High Regional Stewardship Priority:")
print(stewardship_counts)


Unique values in 'regional stewardship status':
[True False]
High Regional Stewardship Priority:
False    6101
True     2445
Name: regional_stewardship_status, dtype: int64


In [85]:
#Q9: Which species are most often seen in flyovers? 

species_col = 'common_name'
flyover_col = 'flyover_observed'

# Filter for flyover observations only
flyover_df = forest_df[forest_df[flyover_col] == True]

# Count species in flyovers
flyover_species_counts = flyover_df[species_col].value_counts().head(10)

# Display result
print("Top 10 Species Seen in Flyovers:")
print(flyover_species_counts)


Top 10 Species Seen in Flyovers:
Common Grackle        17
American Crow         13
Cedar Waxwing          8
American Goldfinch     6
Canada Goose           5
Blue Jay               5
Mourning Dove          4
Turkey Vulture         4
Great Blue Heron       4
Chimney Swift          3
Name: common_name, dtype: int64


In [87]:
#Q10: What is the sex-wise distribution for the top 5 species? 

# Define columns
species_col = 'common_name'
sex_col = 'sex_category'

# Step 1: Get top 5 most observed species
top_species = forest_df[species_col].value_counts().head(5).index.tolist()

# Step 2: Filter data to only include top 5 species
top_species_df = forest_df[forest_df[species_col].isin(top_species)]

# Step 3: Count sex category per species
sex_distribution = top_species_df.groupby([species_col, sex_col]).size().unstack(fill_value=0)

# Display result
print(" Sex-wise Distribution for Top 5 Species:")
print(sex_distribution)


 Sex-wise Distribution for Top 5 Species:
sex_category             Male  Undetermined
common_name                                
Carolina Wren               0           646
Eastern Tufted Titmouse     0           541
Eastern Wood-Pewee          0           486
Northern Cardinal          25           570
Red-eyed Vireo              0           694


In [88]:
#Q11: How does observation count vary by sky and wind condition? 

# Define environmental columns
sky_col = 'sky'
wind_col = 'wind'

# Step 1: Group by sky and wind condition and count observations
observation_counts = forest_df.groupby([sky_col, wind_col]).size().unstack(fill_value=0)

# Display result
print("Observation Counts by Sky and Wind Conditions:")
print(observation_counts)


Observation Counts by Sky and Wind Conditions:
wind                 Calm (< 1 mph) smoke rises vertically  \
sky                                                          
Clear or Few Clouds                                   1211   
Cloudy/Overcast                                        696   
Fog                                                    168   
Mist/Drizzle                                            69   
Partly Cloudy                                          946   

wind                 Gentle breeze (8-12 mph), leaves in motion  \
sky                                                               
Clear or Few Clouds                                          45   
Cloudy/Overcast                                              29   
Fog                                                           0   
Mist/Drizzle                                                  0   
Partly Cloudy                                                64   

wind                 Light air movement (1-3 mph

In [89]:
#Q12: Does disturbance level affect species count? 

# Define columns
disturbance_col = 'disturbance'
species_col = 'common_name'

# Group by disturbance and count unique species
species_by_disturbance = forest_df.groupby(disturbance_col)[species_col].nunique()

# Display result
print("Unique Species Count by Disturbance Level:")
print(species_by_disturbance)


Unique Species Count by Disturbance Level:
disturbance
Moderate effect on count    65
No effect on count          96
Serious effect on count     45
Slight effect on count      91
Name: common_name, dtype: int64


In [97]:
#Q13: Which weather combinations (sky + wind + temperature) support highest species richness? 

# Define columns
species_col = 'common_name'
weather_col = 'weather_combination'

# Drop missing values
weather_df = forest_df.dropna(subset=[species_col, weather_col])

# Group by weather condition and count unique species
richness_by_weather = weather_df.groupby(weather_col)[species_col].nunique().sort_values(ascending=False)

# Display result
print("Species Richness by Weather Combination:")
print(richness_by_weather)



Species Richness by Weather Combination:
weather_combination
Clear or Few Clouds | Light air movement (1-3 mph) smoke drifts | Temp: 18.39999962    28
Partly Cloudy | Light air movement (1-3 mph) smoke drifts | Temp: 22.0                 27
Partly Cloudy | Light air movement (1-3 mph) smoke drifts | Temp: 21.29999924          25
Clear or Few Clouds | Light air movement (1-3 mph) smoke drifts | Temp: 20.39999962    24
Clear or Few Clouds | Calm (< 1 mph) smoke rises vertically | Temp: 22.0               23
                                                                                       ..
Partly Cloudy | Light air movement (1-3 mph) smoke drifts | Temp: 24.0                  3
Partly Cloudy | Light air movement (1-3 mph) smoke drifts | Temp: 20.10000038           3
Clear or Few Clouds | Light breeze (4-7 mph) wind felt on face | Temp: 21.0             3
Clear or Few Clouds | Light breeze (4-7 mph) wind felt on face | Temp: 28.39999962      3
Clear or Few Clouds | Light breeze (4-7

In [98]:
#Q14: Do certain observers report flyovers more frequently in specific plots? 

# Define columns
species_col = 'common_name'
activity_col = 'plot_activity_score'
disturbance_col = 'disturbance'

# Drop missing values
plot_df = forest_df.dropna(subset=[species_col, activity_col, disturbance_col])

# Group by plot activity and disturbance, count unique species
richness_by_activity_disturbance = (
    plot_df.groupby([activity_col, disturbance_col])[species_col]
    .nunique()
    .unstack(fill_value=0)
)

# Display result
print("Species Richness by Plot Activity and Disturbance:")
print(richness_by_activity_disturbance)


Species Richness by Plot Activity and Disturbance:
disturbance          Moderate effect on count  No effect on count  \
plot_activity_score                                                 
5.0                                         0                   0   
6.0                                         0                   4   
8.0                                         0                   0   
9.0                                         3                   0   
10.0                                        9                   5   
11.0                                       21                  18   
12.0                                        0                  15   
13.0                                        0                  30   
14.0                                       26                  34   
15.0                                       23                  28   
16.0                                       19                  34   
17.0                                       20       

In [101]:
#Q15: Is sex-based observation or ID method influenced by seasonality? 

# Drop missing values
season_sex_df = forest_df.dropna(subset=['season', 'sex_category'])

# Count observations by season and sex
sex_by_season = season_sex_df.groupby(['season', 'sex_category']).size().unstack(fill_value=0)

# Display result
print("Sex-Based Observations by Season:")
print(sex_by_season)

# Drop missing values
season_id_df = forest_df.dropna(subset=['season', 'id_method'])

# Count observations by season and ID method
id_by_season = season_id_df.groupby(['season', 'id_method']).size().unstack(fill_value=0)

# Display result
print("ID Method Usage by Season:")
print(id_by_season)


Sex-Based Observations by Season:
sex_category  Male  Undetermined
season                          
Autumn           5          1173
Spring          26          2950
Summer          19          3303
Winter           9          1061
ID Method Usage by Season:
id_method  Calling  Singing  Visualization
season                                    
Autumn         391      747             40
Spring         892     1910            174
Summer        1021     2134            166
Winter         373      639             58


In [102]:
#Q16: Who are the top observers by number of records? 

# Define relevant columns
species_col = 'common_name'
flyover_col = 'flyover_observed'
distance_col = 'distance_category'

# Drop missing values
df = forest_df.dropna(subset=[species_col, flyover_col, distance_col])

# Group by flyover + distance and count unique species
richness_flyover_distance = (
    df.groupby([flyover_col, distance_col])[species_col]
    .nunique()
    .unstack(fill_value=0)
)

# Display result
print("Species Richness by Flyover and Distance Category:")
print(richness_flyover_distance)

Species Richness by Flyover and Distance Category:
distance_category  >50m
flyover_observed       
False               106
True                 23


In [105]:
#Q17: Is there a pattern between observer + habitat type + conservation category? 

# Drop rows with missing key info
df = forest_df.dropna(subset=['observer', 'location_type', 'regional_stewardship_status'])

# Group by all three and count how many records per combination
observer_habitat_conservation = (
    df.groupby(['observer', 'location_type', 'regional_stewardship_status'])
    .size()
    .reset_index(name='observation_count')
)

# Display result
print("Observer × Habitat × Conservation Category Counts:")
print(observer_habitat_conservation.head(10))  # preview top 10 rows


Observer × Habitat × Conservation Category Counts:
           observer location_type  regional_stewardship_status  \
0    Brian Swimelar        Forest                        False   
1    Brian Swimelar        Forest                         True   
2  Elizabeth Oswald        Forest                        False   
3  Elizabeth Oswald        Forest                         True   
4    Kimberly Serno        Forest                        False   
5    Kimberly Serno        Forest                         True   

   observation_count  
0               1679  
1                672  
2               2334  
3                951  
4               2088  
5                822  


In [ ]:
#BIRD_SPECIES_OBSERVATION_GRASSLAND

In [136]:
#Q1: Sex-wise Distribution for Top 5 Species

for name, df in enhanced_grassland_dfs.items():
    if 'common_name' in df.columns and 'sex_category' in df.columns:
        top_species = df['common_name'].value_counts().head(5).index.tolist()
        filtered_df = df[df['common_name'].isin(top_species)]
        
        print(f"\n{name} - Sex Distribution for Top 5 Species:")
        result = filtered_df.groupby(['common_name', 'sex_category']).size().unstack(fill_value=0)
        print(result)
    else:
        print(f"\n{name}: Required column(s) missing.")



ANTI_GRASSLAND - Sex Distribution for Top 5 Species:
sex_category         Female  Male  Undetermined
common_name                                    
Chipping Sparrow          0   185            19
Field Sparrow             0   289             6
Grasshopper Sparrow       0   160            10
Indigo Bunting            4   191             4
Northern Cardinal         9    25           233

HAFE_GRASSLAND - Sex Distribution for Top 5 Species:
sex_category          Male  Undetermined
common_name                             
Field Sparrow            9             1
Grasshopper Sparrow      7             1
Northern Cardinal        1             6
Red-winged Blackbird    10             1
Song Sparrow             8             0

MANA_GRASSLAND - Sex Distribution for Top 5 Species:
sex_category          Female  Male  Undetermined
common_name                                     
Eastern Meadowlark         2    81            14
Field Sparrow              0   113             1
Indigo Bunting     

In [133]:
#Q2: Most frequently observed species in each grassland dataset

for name, df in enhanced_grassland_dfs.items():
    if 'common_name' in df.columns:
        print(f"\n{name} - Top 10 Observed Species:")
        print(df['common_name'].value_counts().head(10))
    else:
        print(f"\n{name}: 'common_name' column missing.")



ANTI_GRASSLAND - Top 10 Observed Species:
Field Sparrow          295
Northern Cardinal      267
Chipping Sparrow       204
Indigo Bunting         199
Grasshopper Sparrow    170
Mourning Dove          149
American Goldfinch     144
Carolina Wren          130
Eastern Bluebird       101
Eastern Towhee          99
Name: common_name, dtype: int64

HAFE_GRASSLAND - Top 10 Observed Species:
Red-winged Blackbird    11
Field Sparrow           10
Song Sparrow             8
Grasshopper Sparrow      8
Northern Cardinal        7
Brown-headed Cowbird     6
Indigo Bunting           6
American Goldfinch       5
Northern Mockingbird     5
House Wren               4
Name: common_name, dtype: int64

MANA_GRASSLAND - Top 10 Observed Species:
Field Sparrow           114
Northern Cardinal       111
Red-winged Blackbird    102
Eastern Meadowlark       97
Indigo Bunting           87
Grasshopper Sparrow      86
Carolina Wren            72
American Goldfinch       62
Common Yellowthroat      61
Eastern Towhee 

In [131]:
#Q3: Distribution of Bird Identification Methods

for name, df in enhanced_grassland_dfs.items():
    if 'id_method' in df.columns:
        print(f"\n{name} - ID Method Distribution:")
        print(df['id_method'].value_counts(dropna=False))
    else:
        print(f"\n{name}: 'id_method' column missing.")



ANTI_GRASSLAND - ID Method Distribution:
Singing          1989
Calling           586
Visualization     554
NaN                 1
Name: id_method, dtype: int64

HAFE_GRASSLAND - ID Method Distribution:
Singing          75
Calling          19
Visualization    14
NaN               1
Name: id_method, dtype: int64

MANA_GRASSLAND - ID Method Distribution:
Singing          931
Visualization    254
Calling          251
Name: id_method, dtype: int64

MONO_GRASSLAND - ID Method Distribution:
Singing          1196
Visualization     548
Calling           408
Name: id_method, dtype: int64


In [123]:
#Q4: Distribution of Sex Categories

for name, df in enhanced_grassland_dfs.items():
    if 'sex_category' in df.columns:
        print(f"\n{name} - Sex Category Distribution:")
        print(df['sex_category'].value_counts(dropna=False))
    else:
        print(f"\n{name}: 'sex_category' column missing.")



ANTI_GRASSLAND - Sex Category Distribution:
Undetermined    1736
Male            1348
Female            46
Name: sex_category, dtype: int64

HAFE_GRASSLAND - Sex Category Distribution:
Male            55
Undetermined    53
Female           1
Name: sex_category, dtype: int64

MANA_GRASSLAND - Sex Category Distribution:
Male            753
Undetermined    658
Female           25
Name: sex_category, dtype: int64

MONO_GRASSLAND - Sex Category Distribution:
Undetermined    1204
Male             894
Female            54
Name: sex_category, dtype: int64


In [128]:
#Q5: Top Weather Condition Combinations

for name, df in enhanced_grassland_dfs.items():
    if 'weather_combination' in df.columns:
        print(f"\n{name} - Top Weather Combinations:")
        print(df['weather_combination'].value_counts(dropna=False).head(10))
    else:
        print(f"\n{name}: 'weather_combination' column missing.")



ANTI_GRASSLAND - Top Weather Combinations:
Cloudy/Overcast | Light breeze (4-7 mph) wind felt on face | Temp: 23.89999962     63
Partly Cloudy | Light air movement (1-3 mph) smoke drifts | Temp: 21.39999962      49
Partly Cloudy | Light air movement (1-3 mph) smoke drifts | Temp: 22.0             47
Partly Cloudy | Light air movement (1-3 mph) smoke drifts | Temp: 21.60000038      37
Partly Cloudy | Light air movement (1-3 mph) smoke drifts | Temp: 21.20000076      37
Cloudy/Overcast | Light air movement (1-3 mph) smoke drifts | Temp: 22.0           34
Cloudy/Overcast | Light air movement (1-3 mph) smoke drifts | Temp: 19.79999924    34
Partly Cloudy | Light air movement (1-3 mph) smoke drifts | Temp: 21.0             33
Cloudy/Overcast | Light air movement (1-3 mph) smoke drifts | Temp: 22.60000038    33
Partly Cloudy | Light air movement (1-3 mph) smoke drifts | Temp: 26.5             32
Name: weather_combination, dtype: int64

HAFE_GRASSLAND - Top Weather Combinations:
Cloudy/Overc

In [129]:
#Q6: Distance Category Distribution (>50m vs ≤50m)

for name, df in enhanced_grassland_dfs.items():
    if 'distance_category' in df.columns:
        print(f"\n{name} - Distance Category Distribution:")
        print(df['distance_category'].value_counts(dropna=False))
    else:
        print(f"\n{name}: 'distance_category' column missing.")



ANTI_GRASSLAND - Distance Category Distribution:
<= 50 meters    2940
Unknown          190
Name: distance_category, dtype: int64

HAFE_GRASSLAND - Distance Category Distribution:
<= 50 meters    99
Unknown         10
Name: distance_category, dtype: int64

MANA_GRASSLAND - Distance Category Distribution:
<= 50 meters    1307
Unknown          129
Name: distance_category, dtype: int64

MONO_GRASSLAND - Distance Category Distribution:
<= 50 meters    1883
Unknown          269
Name: distance_category, dtype: int64


In [130]:
#Q7: Count of High Regional Stewardship Priority Birds

for name, df in enhanced_grassland_dfs.items():
    if 'regional_stewardship_status' in df.columns:
        print(f"\n{name} - Regional Stewardship Status Distribution:")
        print(df['regional_stewardship_status'].value_counts(dropna=False))
    else:
        print(f"\n{name}: 'regional_stewardship_status' column missing.")



ANTI_GRASSLAND - Regional Stewardship Status Distribution:
False    2385
True      745
Name: regional_stewardship_status, dtype: int64

HAFE_GRASSLAND - Regional Stewardship Status Distribution:
False    87
True     21
NaN       1
Name: regional_stewardship_status, dtype: int64

MANA_GRASSLAND - Regional Stewardship Status Distribution:
False    1093
True      343
Name: regional_stewardship_status, dtype: int64

MONO_GRASSLAND - Regional Stewardship Status Distribution:
False    1816
True      336
Name: regional_stewardship_status, dtype: int64


In [134]:
#Q8: Top Species Observed in Flyovers

for name, df in enhanced_grassland_dfs.items():
    if 'flyover_observed' in df.columns and 'common_name' in df.columns:
        flyover_species = df[df['flyover_observed'] == True]
        print(f"\n{name} - Top Flyover Species:")
        print(flyover_species['common_name'].value_counts(dropna=False).head(10))
    else:
        print(f"\n{name}: Required column(s) missing.")



ANTI_GRASSLAND - Top Flyover Species:
Common Grackle          31
Barn Swallow            29
American Goldfinch      26
Mourning Dove           18
Turkey Vulture          16
Tree Swallow            11
European Starling        9
Cedar Waxwing            8
American Crow            8
Brown-headed Cowbird     7
Name: common_name, dtype: int64

HAFE_GRASSLAND - Top Flyover Species:
American Goldfinch      3
Brown-headed Cowbird    2
Red-winged Blackbird    1
Unidentified Crow       1
Blue Jay                1
American Robin          1
Name: common_name, dtype: int64

MANA_GRASSLAND - Top Flyover Species:
American Goldfinch               22
Barn Swallow                     19
Cedar Waxwing                    13
Red-winged Blackbird             12
Mourning Dove                     9
European Starling                 7
Common Grackle                    6
Tree Swallow                      6
Northern Rough-winged Swallow     4
Great Blue Heron                  4
Name: common_name, dtype: int64



In [137]:
#Q9:Observation Count by Sky Condition

for name, df in enhanced_grassland_dfs.items():
    if 'sky' in df.columns:
        print(f"\n{name} - Observations by Sky Condition:")
        print(df['sky'].value_counts(dropna=False))
    else:
        print(f"\n{name}: 'sky' column missing.")



ANTI_GRASSLAND - Observations by Sky Condition:
Partly Cloudy          1486
Clear or Few Clouds    1219
Cloudy/Overcast         397
Mist/Drizzle             28
Name: sky, dtype: int64

HAFE_GRASSLAND - Observations by Sky Condition:
Cloudy/Overcast        81
Clear or Few Clouds    27
NaN                     1
Name: sky, dtype: int64

MANA_GRASSLAND - Observations by Sky Condition:
Partly Cloudy          601
Clear or Few Clouds    532
Cloudy/Overcast        150
Fog                    117
Mist/Drizzle            36
Name: sky, dtype: int64

MONO_GRASSLAND - Observations by Sky Condition:
Partly Cloudy          1096
Clear or Few Clouds     579
Cloudy/Overcast         412
Fog                      44
Mist/Drizzle             21
Name: sky, dtype: int64


In [138]:
#Q10:Observation Count by Wind Condition

for name, df in enhanced_grassland_dfs.items():
    if 'wind' in df.columns:
        print(f"\n{name} - Observations by Wind Condition:")
        print(df['wind'].value_counts(dropna=False))
    else:
        print(f"\n{name}: 'wind' column missing.")



ANTI_GRASSLAND - Observations by Wind Condition:
Light air movement (1-3 mph) smoke drifts     1721
Light breeze (4-7 mph) wind felt on face       871
Calm (< 1 mph) smoke rises vertically          416
Gentle breeze (8-12 mph), leaves in motion     122
Name: wind, dtype: int64

HAFE_GRASSLAND - Observations by Wind Condition:
Light breeze (4-7 mph) wind felt on face     65
Light air movement (1-3 mph) smoke drifts    43
NaN                                           1
Name: wind, dtype: int64

MANA_GRASSLAND - Observations by Wind Condition:
Light air movement (1-3 mph) smoke drifts     644
Calm (< 1 mph) smoke rises vertically         393
Light breeze (4-7 mph) wind felt on face      350
Gentle breeze (8-12 mph), leaves in motion     49
Name: wind, dtype: int64

MONO_GRASSLAND - Observations by Wind Condition:
Light air movement (1-3 mph) smoke drifts     1126
Light breeze (4-7 mph) wind felt on face       659
Calm (< 1 mph) smoke rises vertically          311
Gentle breeze (8-12 mph)

In [139]:
#Q11: Species Count by Disturbance Level

for name, df in enhanced_grassland_dfs.items():
    if 'disturbance' in df.columns and 'common_name' in df.columns:
        print(f"\n{name} - Species Count by Disturbance Level:")
        result = df.groupby('disturbance')['common_name'].nunique()
        print(result)
    else:
        print(f"\n{name}: Required column(s) missing.")



ANTI_GRASSLAND - Species Count by Disturbance Level:
disturbance
Moderate effect on count    43
No effect on count          71
Serious effect on count     25
Slight effect on count      71
Name: common_name, dtype: int64

HAFE_GRASSLAND - Species Count by Disturbance Level:
disturbance
Moderate effect on count     6
No effect on count          18
Serious effect on count      3
Slight effect on count      25
Name: common_name, dtype: int64

MANA_GRASSLAND - Species Count by Disturbance Level:
disturbance
Moderate effect on count    36
No effect on count          63
Serious effect on count      4
Slight effect on count      57
Name: common_name, dtype: int64

MONO_GRASSLAND - Species Count by Disturbance Level:
disturbance
Moderate effect on count    62
No effect on count          60
Serious effect on count     39
Slight effect on count      83
Name: common_name, dtype: int64


In [141]:
#Q12: Observer Frequency Distribution

for name, df in enhanced_grassland_dfs.items():
    if 'observer_freq' in df.columns:
        print(f"\n{name} - Observer Frequency Distribution:")
        print(df['observer_freq'].value_counts(dropna=False))
    else:
        print(f"\n{name}: 'observer_freq' column missing.")



ANTI_GRASSLAND - Observer Frequency Distribution:
1163    1163
1087    1087
880      880
Name: observer_freq, dtype: int64

HAFE_GRASSLAND - Observer Frequency Distribution:
44.0    44
35.0    35
29.0    29
NaN      1
Name: observer_freq, dtype: int64

MANA_GRASSLAND - Observer Frequency Distribution:
514    514
492    492
430    430
Name: observer_freq, dtype: int64

MONO_GRASSLAND - Observer Frequency Distribution:
791    791
788    788
573    573
Name: observer_freq, dtype: int64


In [149]:
# Export Grassland Data
enhanced_grassland_df = pd.concat(enhanced_grassland_dfs.values(), ignore_index=True)
enhanced_grassland_df.to_csv(r'C:\Users\addyu\OneDrive\Desktop\LABMENTIX\BIRD SPECIES OBSERVATION\enhanced_grassland.csv', index=False)

# Export Forest Data
enhanced_forest_df = pd.concat(enhanced_forest_dfs.values(), ignore_index=True)
enhanced_forest_df.to_csv(r'C:\Users\addyu\OneDrive\Desktop\LABMENTIX\BIRD SPECIES OBSERVATION\enhanced_forest.csv', index=False)

# Create and Export Master Dataset
combined_df = pd.concat([enhanced_grassland_df, enhanced_forest_df], ignore_index=True)
combined_df.to_csv(r'C:\Users\addyu\OneDrive\Desktop\LABMENTIX\BIRD SPECIES OBSERVATION\master_bird_data.csv', index=False)

print("All files exported successfully.")


All files exported successfully.
